<a href="https://colab.research.google.com/github/Siva22223333/Ego-detection/blob/main/ego_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

DATASET_DIR = "/content/drive/MyDrive/face"
IMG_SIZE = (160, 160)
BATCH_SIZE = 16
EPOCHS = 10
MODEL_OUT = "ego_model.keras" # Changed to .keras

from google.colab import drive
drive.mount('/content/drive')


def build_datasets():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary",
    )

    print("Class order:", train_ds.class_names)

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    return train_ds, val_ds


def build_model():
    data_augmentation = models.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ])

    base_model = MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After running the above cell and authorizing Google Drive access, please ensure that the directory `/content/drive/MyDrive/face` exists and contains your image dataset. If it doesn't, you will need to upload your dataset to that location in your Google Drive.

In [ ]:

def main():
    train_ds, val_ds = build_datasets()
    model = build_model()
    model.summary()

    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

    model.save(MODEL_OUT)
    print(f"Saved trained model to {MODEL_OUT}")


if __name__ == "__main__":
    main()

Found 1963 files belonging to 2 classes.
Using 1571 files for training.
Found 1963 files belonging to 2 classes.
Using 392 files for validation.
Class order: ['egodataset', 'noegodataset']


Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 87s 820ms/step - accuracy: 0.9803 - loss: 0.0517 - val_accuracy: 1.0000 - val_loss: 5.6610e-04
Epoch 2/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 53s 523ms/step - accuracy: 0.9981 - loss: 0.0093 - val_accuracy: 1.0000 - val_loss: 7.7143e-04
Epoch 3/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 84s 549ms/step - accuracy: 0.9981 - loss: 0.0049 - val_accuracy: 1.0000 - val_loss: 8.7096e-04
Epoch 4/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 54s 542ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 1.0000 - val_loss: 6.8685e-04
Epoch 5/10
99/99 ━━━━━━━━━━━━━━━━━━━━ 54s 548ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 1.0000 - val_loss: 4.7521e-04
Epoch 6/10
51/99 ━━━━━━━━━━━━━━━━━━━━ 20s 423ms/step - accuracy: 1.0000 - loss: 5.8828e-04

### Dataset Directory Structure for Binary Classification

For `tf.keras.utils.image_dataset_from_directory` with `label_mode="binary"`, your dataset directory (`/content/drive/MyDrive/face`) must contain exactly two subdirectories. Each subdirectory represents one of your classes and should contain the images belonging to that class.

For example:

```
/content/drive/MyDrive/face/
├── class_0_name/
│   ├── image_001.jpg
│   ├── image_002.png
│   └── ...
└── class_1_name/
    ├── image_A.jpg
    ├── image_B.png
    └── ...
```

Please verify that your `DATASET_DIR` adheres to this structure. If not, you will need to organize your images accordingly in Google Drive.

In [ ]:
import os

# Check if the DATASET_DIR exists
if not os.path.exists(DATASET_DIR):
    print(f"Error: DATASET_DIR '{DATASET_DIR}' does not exist. Please ensure your Google Drive is mounted and the path is correct.")
else:
    print(f"Contents of {DATASET_DIR}:")
    # List subdirectories (which should be your class names)
    subdirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
    if not subdirs:
        print("No subdirectories (classes) found. Please organize your images into two class-specific subfolders.")
    else:
        print("Found subdirectories (classes):")
        for subdir in subdirs:
            print(f"- {subdir}")
            # Optionally, print a few files from each subdirectory to confirm content
            files = os.listdir(os.path.join(DATASET_DIR, subdir))
            if files:
                print(f"  First 3 files in {subdir}: {files[:3]}")
            else:
                print(f"  (No files found in {subdir})")

Contents of /content/drive/MyDrive/face:
Found subdirectories (classes):
- noegodataset
  First 3 files in noegodataset: ['Copy of IMG_20260731_211254.jpg', 'Copy of IMG_20260731_211304.jpg', 'Copy of IMG_20260731_211258.jpg']
- egodataset
  First 3 files in egodataset: ['disgust_0_8576.jpeg', 'disgust_0_8617.jpeg', 'disgust_0_7617.jpeg']


In [ ]:
import cv2
import time
import numpy as np
import tensorflow as tf

# -----------------------------
# Configuration
# -----------------------------
MODEL_PATH = "ego_model.keras" # Updated to .keras
SOUND_PATH = "/content/drive/MyDrive/faaah.wav"
IMG_SIZE = (160, 160)
CONFIDENCE_THRESHOLD = 0.75
COOLDOWN_SECONDS = 3

# -----------------------------
# Load model
# -----------------------------
# Using .keras format often avoids the need for custom_object_scope for standard applications
try:
    model = tf.keras.models.load_model(MODEL_PATH)
except Exception:
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    model = tf.keras.models.load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input, 'TrueDivide': tf.math.truediv})

print("Model loaded successfully.")

# -----------------------------
# Face detector
# -----------------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise RuntimeError("Cannot load Haar Cascade XML")

# -----------------------------
# Prediction
# -----------------------------
def predict(face):
    face = cv2.resize(face, IMG_SIZE)
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face = face.astype("float32")
    face = np.expand_dims(face, axis=0)
    probability = model.predict(face, verbose=0)[0][0]
    return probability

print("Setup complete.")

Model loaded successfully.
Setup complete.


### Colab-Compatible Webcam and Sound Solution
Since `cv2.VideoCapture` doesn't work in the cloud, we use JavaScript to capture frames from your local browser's webcam and send them to the Python kernel for processing.

In [ ]:
from google.colab.output import eval_js
from IPython.display import display, Javascript, Audio, Image
from base64 import b64decode
import cv2
import numpy as np

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      div.style.border = '2px solid #4CAF50';
      div.style.padding = '10px';
      div.style.width = 'max-content';

      const capture = document.createElement('button');
      capture.textContent = 'CLICK HERE TO CAPTURE';
      capture.style.background = '#4CAF50';
      capture.style.color = 'white';
      capture.style.fontSize = '18px';
      capture.style.fontWeight = 'bold';
      capture.style.padding = '15px';
      capture.style.margin = '10px';
      capture.style.cursor = 'pointer';
      capture.style.display = 'block';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.width = '400px';

      try {
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        const result = await new Promise((resolve) => {
          capture.onclick = () => {
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            resolve(canvas.toDataURL('image/jpeg', quality));
          };
        });
        return result;
      } catch (err) {
        div.remove();
        console.error(err);
        return null;
      }
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))

  if data is None:
      return None

  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

try:
  print("-- WEBCAM INTERFACE STARTING ---")
  print("1. Look for the 'Allow' prompt at the top of your browser.")
  print("2. Click the large green 'CLICK HERE TO CAPTURE' button.")

  filename = take_photo()
  if filename is None:
      print("\n[!] Camera failed. Please ensure you are using HTTPS and allowed camera access.")
  else:
      print(f'\nPhoto taken!')
      display(Image(filename))

      img = cv2.imread(filename)
      gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

      # High sensitivity parameters
      faces = face_cascade.detectMultiScale(gray, scaleFactor=1.05, minNeighbors=2, minSize=(30, 30))

      if len(faces) == 0:
          print("No face detected in the picture. Try again with more light.")
      else:
          print(f"Found {len(faces)} face(s). Checking for Ego...")

      for (x, y, w, h) in faces:
          face_crop = img[y:y+h, x:x+w] # Define face_crop here
          prob_noego = predict(face_crop) # This is the probability of being 'noegodataset' (class 1)
          prob_ego = 1 - prob_noego      # This is the probability of being 'egodataset' (class 0)

          if prob_ego >= CONFIDENCE_THRESHOLD:
              print(f"!!! EGO DETECTED ({prob_ego*100:.1f}%) !!!")
              try:
                  display(Audio(SOUND_PATH, autoplay=True))
              except: pass
          else:
              print(f"Status: Normal ({prob_noego*100:.1f}%) ") # If not ego, it's normal. Display confidence for being normal.

except Exception as err:
  print(f"Error: {err}")

-- WEBCAM INTERFACE STARTING ---
1. Look for the 'Allow' prompt at the top of your browser.
2. Click the large green 'CLICK HERE TO CAPTURE' button.


<IPython.core.display.Javascript object>


[!] Camera failed. Please ensure you are using HTTPS and allowed camera access.


In [ ]:
import cv2

print(cv2.__version__)
print(hasattr(cv2, "CascadeClassifier"))

4.10.0
True


In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install opencv-python-headless==4.10.0.84

Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.9 MB)
